# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
        check=True
    )
else:
    # Find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Working dir:", os.getcwd())
print("Starter dataset loaded:", df.shape)

Working dir: /content/flyrank-ml-internship-starter
Starter dataset loaded: (30000, 44)


In [5]:
import os
from getpass import getpass

HF_TOKEN = getpass("Enter the access token: ")

if not HF_TOKEN:
    print("HF_TOKEN was not provided.")
else:
    print("HF_TOKEN found.")

Enter the access token: ··········
HF_TOKEN found.


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## 1. Unit of analysis + time window

For my Refresh / Content Opportunity Scoring lane, the unit of analysis is **one page observation for a defined reporting month**.

I will develop and verify the data contract using a **mid-panel month, March 2026 (`month = 2026-03`)**. The monthly observation provides search-performance and content-related signals that can be used to prioritize pages for review.

I will avoid using the `_sample` table for developing label logic because the assignment identifies it as the final June 2026 month and therefore a natural outcome/test window.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## 2. Fields: feature / label / context / excluded

### Features

I will start with five candidate features:

* **`search_volume`** — available at the decision moment because it represents the search demand associated with the page's search context.
* **`impressions_90d`** — available at the decision moment because it summarizes observed search visibility over the preceding period.
* **`ctr`** — available at the decision moment because it is an observed search-performance signal.
* **`avg_position`** — available at the decision moment because it describes observed search ranking performance.
* **`content_age_days`** — available at the decision moment because the age of the content is known when deciding which pages to review.

### Label / proxy

For the initial framing, the outcome of interest is **declining performance**, represented by the observed `trend_direction` outcome.

### Context

The month, page identity, and other descriptive fields provide context for interpreting the observations and validating the data.

### Deliberately excluded

I will exclude `trend_direction` and `trend_pct` from model features because they are directly connected to the declining-performance outcome and can create target leakage. I will also exclude any information that would only become known after the decision point.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import duckdb

con = duckdb.connect()

# Use the Hugging Face token securely
con.execute(
    "CREATE SECRET (TYPE huggingface, TOKEN ?)",
    [HF_TOKEN]
)

rel = "hf://datasets/FlyRank/internship-warehouse"

print("DuckDB connected to Hugging Face warehouse.")

DuckDB connected to Hugging Face warehouse.


In [11]:
con.sql("""
    SELECT COUNT(*)
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     78835655 │
└──────────────┘



## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## 4. Data limits

This data can support observations about the available search-performance and content signals, but it cannot establish causation.

The history may be unbalanced across pages, and the available fields may differ depending on the period. Early observations may contain more limited Search Console information. Time windows can also overlap, so observations from nearby months should not automatically be treated as independent experiments.

The final month should be treated separately from development because it represents the natural future outcome window. I will therefore use a mid-panel month such as March 2026 while developing the contract and avoid using the final `_sample` month to develop label logic.

A further limitation is that a model or ranking can identify pages that resemble the observed outcome, but it cannot by itself determine whether a specific content action will cause performance to improve. Human review is still required.


In [12]:
march_check = con.sql("""
    SELECT *
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
    LIMIT 5
""").df()

display(march_check)
print("Columns:")
print(march_check.columns.tolist())

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


Columns:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


In [13]:
grain_check = con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT client_hash_id) AS unique_clients,
        COUNT(DISTINCT content_hash_id) AS unique_content_items,
        COUNT(DISTINCT CONCAT(client_hash_id, '||', content_hash_id)) AS unique_client_content_pairs
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
""").df()

display(grain_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_clients,unique_content_items,unique_client_content_pairs
0,9841378,55,331437,331437


In [14]:
slice_check = con.sql("""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS start_date,
        MAX(report_date) AS end_date
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
""").df()

display(slice_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,start_date,end_date
0,9841378,2026-03-01,2026-03-31


In [15]:
availability_check = con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
""").df()

display(availability_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available_rows,ga4_available_rows
0,9841378,3611061,413966


In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_sum_position",
    "scroll_events"
]

feature_frame = march_check[feature_cols].copy()

display(feature_frame.head())

print("Shape:", feature_frame.shape)

,gsc_impressions,gsc_clicks,gsc_sum_position,scroll_events
0,20,0,67,<NA>
1,1,0,0,<NA>
2,125,1,616,<NA>
3,7,0,28,<NA>
4,11,0,25,<NA>


Shape: (5, 4)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.